# Reach with `robotdesigner` (CPU MuJoCo backend)

A robot arm is *data*; the environment is a TorchRL `EnvBase` built from one declarative `EnvConfig`.
This notebook builds a three-link arm, runs it, and then runs the **same task** on other embodiments.

Install first: `pip install -e ".[env]"` (add `ipykernel` or `jupyter` to run it).

## 1. Describe the robot

`chain()` generates a `ParametricArm`. Every dynamic parameter is explicit; `n` is a generator argument, not a config field.

In [ ]:
import torch

from robotdesigner.config import (
    ControlConfig, EnvConfig, FixedBox, FixedGains, InertiaScaledGains, MujocoOptions,
    OwnReachable, Pose, ReachParams, SimConfig, TaskConfig, chain,
)
from robotdesigner.config import kinematics as kin
from robotdesigner.envs import make_env

arm = chain(
    n=3, axes="zyz", total_length=0.9, total_mass=3.0, radius=0.03,
    damping=0.5, armature=0.01, frictionloss=0.0, effort_limit=50.0, range=(-2.5, 2.5),
    base_pose=Pose((0.0, 0.0, 0.5), (1.0, 0.0, 0.0, 0.0)),   # keep the arm above the floor
)
print("n =", arm.n, "| config_hash =", arm.config_hash)

The numpy-only kinematics oracle gives the end-effector position without any simulator. At `q = 0` the arm is straight along +x.

In [ ]:
import numpy as np

print(kin.forward(arm, np.zeros(3)).ee[:3, 3])          # (0.9, 0, 0.5)
print(kin.forward(arm, [0.5, -0.3, 0.2]).ee[:3, 3])

## 2. Describe the environment

Control, task, simulation and backend options all live in `EnvConfig`, and it has one hash. There are no hidden defaults.

In [ ]:
def make_config(arm, num_envs=4, mode="joint_pos_abs", gains=None, action_scale=None,
                region=None, horizon=100):
    return EnvConfig(
        robot=arm,
        control=ControlConfig(mode, gains or FixedGains(kp=60.0, kd=6.0), action_scale),
        task=TaskConfig(
            id="native/reach", horizon=horizon, success_tolerance=0.02,
            target_region=region or FixedBox(low=(0.2, -0.3, -0.2), high=(0.6, 0.3, 0.2)),
            params=ReachParams(reset_noise=0.0, terminate_on_success=False),
        ),
        sim=SimConfig(backend="mujoco", num_envs=num_envs, device="cpu", dt=0.002,
                      integrator="euler", control_freq=50, seed=0, scene="plain_floor"),
        backend_options=MujocoOptions(self_collision=False),
    )

cfg = make_config(arm)
print("env_hash =", cfg.env_hash)
assert EnvConfig.from_dict(cfg.to_dict()) == cfg      # configs round-trip through plain dicts

## 3. Make the environment

`make_env` checks the config against the backend's capabilities first and reports **every** problem at once.

In [ ]:
env = make_env(cfg)
print(env.batch_size, env.device, env.robot_hash, env.env_hash)
print(env.action_spec.shape, env.observation_spec["proprio", "qpos"].shape)

In [ ]:
from dataclasses import replace

try:
    make_env(replace(cfg, sim=replace(cfg.sim, device="cuda:0"),
                     control=ControlConfig("ee_pos_delta", FixedGains(60.0, 6.0), None)))
except Exception as e:
    print(e)

## 4. Step by hand

The env never autoresets. `step_and_maybe_reset` is the driver-side autoreset; `_reset` masks select which rows to reset.

In [ ]:
td = env.reset()
print("target (base frame):\n", td["task", "target_pos"])

td["action"] = env.action_spec.rand()          # [num_envs, n] in [-1, 1]
td = env.step(td)
print("reward:", td["next", "reward"].flatten())
print("episode_step:", td["next", "episode_step"].flatten())

In [ ]:
# Partial reset: only env 0 returns to the home pose.
td = td["next"].clone()
td["_reset"] = torch.tensor([[True], [False], [False], [False]])
td = env.reset(td)
print(td["episode_step"].flatten())            # env 0 -> 0, others unchanged

## 5. Roll out a random policy

In [ ]:
traj = env.rollout(50)
print(traj.shape)
print("final distance per env:", -traj["next", "reward"][:, -1].flatten())

## 6. One task, several embodiments

The task never mentions `n` or link lengths. Here the same `TaskConfig` runs on arms with 2, 3 and 4 joints and the same total length and mass. `OwnReachable` samples targets each arm can actually reach.

In [ ]:
def reach_arm(n):
    return chain(n=n, axes=("zy" * n)[:n], total_length=0.9, total_mass=3.0, radius=0.03,
                 damping=0.5, armature=0.01, frictionloss=0.0, effort_limit=50.0, range=(-2.5, 2.5),
                 base_pose=Pose((0.0, 0.0, 0.5), (1.0, 0.0, 0.0, 0.0)))

for n in (2, 3, 4):
    e = make_env(make_config(reach_arm(n), region=OwnReachable()))
    t = e.rollout(30)
    print(f"n={n}  robot_hash={e.robot_hash}  env_hash={e.env_hash}  "
          f"action dim={e.action_spec.shape[-1]}  mean final dist={-t['next', 'reward'][:, -1].mean():.3f}")

## 7. Other control modes and inertia-scaled gains

`joint_pos_delta` integrates the action into a clipped position target; `joint_vel` tracks a velocity. With
`InertiaScaledGains` the gains follow the inertia each joint sees (evaluated at `home_qpos`), so differences between
embodiments are not just mismatched gains. The effective gains are recorded on the interface.

In [ ]:
cfg_delta = make_config(arm, mode="joint_pos_delta", action_scale=0.05,
                        gains=InertiaScaledGains(bandwidth=20.0, damping_ratio=1.0))
env_delta = make_env(cfg_delta)
print("kp =", env_delta.interface.kp.round(2))
print("kd =", env_delta.interface.kd.round(2))

td = env_delta.reset()
for _ in range(3):
    td["action"] = torch.ones(4, 3)
    td = env_delta.step(td)["next"]
print("q_target after 3 steps:", env_delta.q_target[0])        # 3 * 0.05 per joint

## 8. Rendering (optional)

Needs an OpenGL context. On a headless machine, set `MUJOCO_GL=egl` before importing mujoco (restart the kernel). Nothing else in the package needs rendering.

In [ ]:
try:
    from robotdesigner.backends.mujoco.render import render_rgb
    import matplotlib.pyplot as plt

    env.reset()
    plt.imshow(render_rgb(env, env_index=0, size=(240, 320)))
    plt.axis("off")
except Exception as e:                  # no GL context, or matplotlib missing
    print("rendering unavailable:", type(e).__name__, e)